<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Production Deployment
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Deployment ist hier kein generisches Docker/FastAPI-Kapitel, sondern der Betrieb eines **kontrollierten Agentensystems**: Monitoring muss Tool-Wahl, Quellenbindung und HITL-Eskalationen sichtbar machen, nicht nur Uptime und Latenz. Ein Meeting- & Research-Briefing-Agent, der im Betrieb unbeobachtet spekuliert, hat sein Leitplanken-Versprechen verloren, selbst wenn er technisch stabil läuft.

Dieses Modul verlässt bewusst den Notebook-Kontext und zeigt dieselbe kontrollierte Agentenidee als betreibbaren Service mit API, Logging und Resilienz.

📦 **Hinweis zum Modul `genai_lib.briefing_rag`:** Sowohl die Notebook-Demo (`prod_agent`) als auch `agent_server.py` nutzen jetzt dieselbe produktive Chroma-Kette aus `genai_lib.briefing_rag` statt eigener Chroma-Duplikate. Im Notebook wird dafür Google Drive gemountet (Colab-Kontext); `agent_server.py` läuft dagegen in einem Server-/Docker-Kontext ohne Colab und liest den Chroma-Datenordner stattdessen über die Umgebungsvariable `BRIEFING_CHROMA_PATH` (Default `/data/chroma_briefing`) — z. B. als gemounteter Docker-Volume-Export der Drive-Collection aus M14.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastapi uvicorn httpx

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M35-Production-Deployment"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M35_Production_Deployment",
    "tags": ["m35", "production"],
    "metadata": {"notebook": "M35", "version": "1.0"}
}


# 1 | Übersicht
---

<p><font color='black' size="5">
Vom Notebook zur Production
</font></p>

Ein Notebook-Agent und ein Production-Agent lösen dieselbe Aufgaben — aber mit
grundlegend verschiedenen Anforderungen:

| Dimension | Notebook (Entwicklung) | Production |
|-----------|----------------------|------------|
| **API-Keys** | Hardcoded / setup_api_keys() | Umgebungsvariablen (`.env`, Secrets) |
| **Fehler** | Exception → Notebook stoppt | Caught → strukturiertes Logging |
| **Skalierung** | 1 User, manuell | N User, automatisch |
| **Deployment** | Jupyter starten | Docker Container / Cloud |
| **Schnittstelle** | Zelle ausführen | REST API (`/invoke`, `/stream`) |
| **Monitoring** | `print()` | LangSmith + Structured Logging |
| **Kosten** | Ignoriert | Gemessen und begrenzt |


# 2 | Von Notebook zu Production
---


<p><font color='black' size="5">
Production-Checkliste
</font></p>

Bevor ein Agent deployed wird, müssen diese fünf Punkte erfüllt sein:

```
✅ 1. API-Keys aus Umgebungsvariablen (nie hardcoded)
✅ 2. Strukturiertes Logging (kein print())
✅ 3. Fehlerbehandlung mit Fallback-Antworten
✅ 4. Timeout und Retry-Logik
✅ 5. Observability: LangSmith Tracing aktiv
```




<p><font color='black' size="5">
Pattern: Production Agent Wrapper
</font></p>

```python
**❌ Notebook-Stil**
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
result = agent.invoke({"messages": [...]})
print(result)

**✅ Production-Stil**
import logging
logger = logging.getLogger(__name__)

async def run_agent(query: str) -> str:
    try:
        result = await agent.ainvoke(
            {"messages": [HumanMessage(query)]},
            config={"recursion_limit": 10}
        )
        return result["messages"][-1].content
    except Exception as e:
        logger.error("Agent error", exc_info=True)
        return "Ein Fehler ist aufgetreten."
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Deployment Pipeline</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    NB["📓 Notebook\nEntwicklung"]
    PY["🐍 Python\nModule\n(agent_server.py)"]
    API["🌐 FastAPI\nServer\n(agent_server.py)"]
    DC["🐳 Docker\nContainer"]
    CL["☁️ Cloud\nDeploy"]

    NB -->|Refactor| PY
    PY -->|Wrap| API
    API -->|Containerize| DC
    DC -->|Deploy| CL

    style NB fill:#37474F,color:#fff
    style PY fill:#1565C0,color:#fff
    style API fill:#2E7D32,color:#fff
    style DC fill:#4A148C,color:#fff
    style CL fill:#E65100,color:#fff
'''
mermaid(diagram, width=900)

✏️ FastAPI, Docker, uvicorn

<details>

+ **FastAPI** ist ein modernes Python‑Web‑Framework, mit dem du sehr schnell performante REST‑APIs (z.B. GET /users, POST /items) bauen kannst.
+ **Docker** ist eine Plattform, mit der du Anwendungen (z.B. deine FastAPI‑App) in isolierten „Containern“ ausführst, sodass sie auf jedem System gleich funktionieren – unabhängig von der Installation von Python, Pip‑Packages etc.
+ **Uvicorn** ist ein leichtgewichtiger, asynchroner Webserver für Python, der speziell für das ASGI‑Standard‑Interface (Asynchronous Server Gateway Interface)gedacht ist.

</details>

In [ ]:
#@markdown   <p><font size="4" color='green'>  🏗️ Production-ready Agent</font> </br></p>

import logging
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

from genai_lib.model_config import WORKER
from google.colab import drive
drive.mount("/content/drive")

from genai_lib.briefing_rag import get_briefing_vectorstore, make_suche_wissensdatenbank_tool

# 1. Strukturiertes Logging konfigurieren
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("agent.production")

# 2. LLM + Tools (API-Key kommt aus os.environ, nie hardcoded)
llm = init_chat_model(WORKER)

# Retrieval: dieselbe produktive, auf Google Drive persistierte Chroma-Collection wie in M14.
vectorstore = get_briefing_vectorstore()
if vectorstore._collection.count() == 0:
    logger.warning("Briefing-Collection ist leer — zuerst M14 ausführen, um den Index auf Drive zu erzeugen.")

suche_wissensdatenbank = make_suche_wissensdatenbank_tool(vectorstore)

prod_agent = create_agent(model=llm, tools=[suche_wissensdatenbank])

# 3. Production Wrapper mit Fehlerbehandlung und Logging
async def run_agent(query: str, session_id: str = "default") -> dict:
    '''Production-ready Agent-Aufruf mit Logging und Fehlerbehandlung.'''
    logger.info("Agent-Anfrage | session=%s | query=%s", session_id, query[:80])
    try:
        result = await prod_agent.ainvoke(
            {"messages": [HumanMessage(content=query)]},
            config={"recursion_limit": 10, "run_name": f"prod-{session_id}"}
        )
        answer = result["messages"][-1].content
        logger.info("Agent-Antwort | session=%s | chars=%d", session_id, len(answer))
        return {"status": "ok", "answer": answer, "session_id": session_id}
    except Exception:
        logger.error("Agent-Fehler | session=%s", session_id, exc_info=True)
        return {"status": "error", "answer": "Ein Fehler ist aufgetreten.", "session_id": session_id}

# Test
result = await run_agent("Warum braucht RAG-Evaluation Quellenbindung?", session_id="demo-001")
mprint(f"**Antwort:** {result['answer']}\n\n**Status:** `{result['status']}` | **Session:** `{result['session_id']}`")

<p><font color='black' size="5">
Graph-Level Resilience: Timeout & Fallback nativ in LangGraph (v1.2.0)
</font></p>

`run_agent()` oben löst Checklisten-Punkt 3 und 4 — Fehlerbehandlung mit Fallback-Antworten, Timeout und Retry-Logik — generisch über `try/except`: jeder Fehler landet in derselben statischen Antwort, ein echtes Timeout gibt es nicht. Das GenAI-Modul **M17_Model_Router** zeigt dasselbe Grundproblem für einzelne LLM-Aufrufe, dort noch vollständig hand-gecodet: Fehler werden in Kategorien (DEAD/TRANSIENT/OURS) einsortiert, ein Circuit Breaker sperrt tote Modelle temporär, ein Fallback übernimmt.

Seit **LangGraph v1.2.0** gibt es dafür native Node-Parameter, die genau dieses Muster ohne Hand-Code abbilden:

| M17-Konzept (Python, hand-gecodet) | LangGraph v1.2.0 (deklarativ) |
|---|---|
| Timeout pro Modellaufruf | `timeout=` in `add_node()` → `NodeTimeoutError` |
| Fehlerklassifikation + Fallback | `error_handler=` in `add_node()` → `NodeError`, Routing per `Command` |
| Circuit Breaker (Cooldown) | Fallback-Node dauerhaft aktivierbar, kombinierbar mit `set_node_defaults()` |

> Referenz: `_docs/LangGraph_Best_Practices.md`, Abschnitt „Advanced: Per-Node-Timeouts" / „Advanced: Node-Level Error-Handler" (Must-Have #2).

In [ ]:
#@markdown   <p><font size="4" color='green'>  🛡️ Resilienter Agent als StateGraph</font> </br></p>

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command, NodeError
from langchain_core.messages import AIMessage

class ResilientAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    fallback_used: bool

def call_prod_agent(state: ResilientAgentState) -> dict:
    """Wrappt den bestehenden prod_agent als einzelnen Node."""
    result = prod_agent.invoke({"messages": state["messages"]})
    return {"messages": result["messages"], "fallback_used": False}

def agent_error_handler(error: NodeError) -> Command:
    """Analog zu M17s DEAD/OURS-Fall: Fehler abfangen statt Absturz, Fallback-Node übernimmt."""
    logger.error("Node '%s' fehlgeschlagen: %s", error.node, error.exception)
    return Command(goto="fallback", update={"fallback_used": True})

def fallback_node(state: ResilientAgentState) -> dict:
    return {"messages": [AIMessage(content="Der Agent ist aktuell nicht erreichbar. Bitte später erneut versuchen.")]}

resilience_graph = StateGraph(ResilientAgentState)
resilience_graph.add_node(
    "agent",
    call_prod_agent,
    timeout=15,                          # Sekunden — ersetzt Checklisten-Punkt 4 (Timeout/Retry)
    error_handler=agent_error_handler,   # ersetzt Checklisten-Punkt 3 (Fallback-Antworten)
)
resilience_graph.add_node("fallback", fallback_node)
resilience_graph.add_edge(START, "agent")
resilience_graph.add_edge("agent", END)
resilience_graph.add_edge("fallback", END)

resilient_agent = resilience_graph.compile()

mprint("✅ Resilienter Agent kompiliert — Timeout 15s, Fallback-Node bei Fehler aktiv")

In [ ]:
#@markdown   <p><font size="4" color='green'>  🧪 Resilienz-Demo: erzwungener Ausfall</font> </br></p>

from langchain_core.messages import HumanMessage

# Normalfall: Agent antwortet regulär, kein Fallback
result_ok = resilient_agent.invoke(
    {"messages": [HumanMessage(content="Was ist LangSmith?")], "fallback_used": False}
)
mprint(f"**Normalfall** — fallback_used: `{result_ok['fallback_used']}`\n\n{result_ok['messages'][-1].content}")

# Erzwungener Ausfall — analog zu M17s FORCED_DEAD_MODELS
def _erzwungener_ausfall(*args, **kwargs):
    raise RuntimeError("Simulierter Provider-Ausfall")

original_invoke = prod_agent.invoke
prod_agent.invoke = _erzwungener_ausfall

result_fail = resilient_agent.invoke(
    {"messages": [HumanMessage(content="Was ist LangGraph?")], "fallback_used": False}
)
mprint(f"**Erzwungener Ausfall** — fallback_used: `{result_fail['fallback_used']}`\n\n{result_fail['messages'][-1].content}")

prod_agent.invoke = original_invoke  # Wiederherstellen

# 2b | Zentrale Modell-Konfiguration
---


<p><font color='black' size="5">
Warum eine zentrale Modell-Konfiguration?
</font></p>

In Notebook-Modulen wird jedes Modell direkt initialisiert — das macht die Rollenlogik
sichtbar und ist didaktisch gewollt. In einem Production-System ist das anders:

| Ansatz | Notebook | Production |
|--------|----------|------------|
| Modell-Init | direkt pro Zelle | zentrale Konfigurationsdatei |
| Modellwechsel | jede Zelle anpassen | eine Stelle ändern |
| Rollenlogik | sichtbar, lehrreich | gekapselt, wartbar |

Die Datei `model_config.py` definiert **alle sechs Rollen** an einer einzigen Stelle.
Ändert sich ein Modell (z. B. nach einem Provider-Update), reicht eine Zeile.

> Rollenübersicht: [Modell-Auswahl Guide](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/modellauswahl.html) · [Provider-Modell-Mapping](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/provider-modell-mapping.html)

In [ ]:
%%writefile model_config.py
"""
model_config.py — Zentrale Modell-Konfiguration für Production-Deployments

Alle sechs Rollen an einer Stelle. Modellwechsel erfordert nur eine Änderung hier.
Rollenlogik: https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/modellauswahl.html
"""
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

# Baseline / Demo — schnell, günstig, Grundlagen und erste Tests
baseline_llm = init_chat_model(BASELINE)

# Router / leichter Reasoner — einfache Routing- und Auswahlentscheidungen
router_llm   = init_chat_model(ROUTER)

# Judge / starker Reasoner — Supervisor, Security, Bewertung, Compliance
judge_llm    = init_chat_model(JUDGE)

# Worker / Synthese — hochwertige Text-, RAG- und strukturierte Ausgabe
worker_llm   = init_chat_model(WORKER)

# Coding-Worker — Code-Generierung, Refactoring, technische Agenten-Knoten
coding_llm   = init_chat_model(WORKER)

# Embeddings — Vektorrepräsentationen für Retrieval und RAG (kein Chat-Modell)
embed_model  = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Verwendung von model_config.py</font> </br></p>

# In jedem Production-Modul: Import statt direkter Initialisierung
# from model_config import worker_llm, judge_llm, embed_model

# Beispiel: Agent mit Rollen aus der zentralen Konfiguration
from model_config import baseline_llm, judge_llm, worker_llm, embed_model
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def beispiel_tool(frage: str) -> str:
    """Beantwortet eine einfache Frage."""
    return f"Antwort auf: {frage}"

# Worker-Agent aus zentraler Konfiguration
agent = create_agent(
    model=worker_llm,
    tools=[beispiel_tool],
    system_prompt="Du bist ein hilfreicher Assistent.",
)

from genai_lib.utilities import mprint
zeilen = [
    "## ✅ model_config.py geladen",
    "",
    "| Rolle | Variable | Modell |",
    "|-------|----------|--------|",
    "| Baseline / Demo | `baseline_llm` | `gpt-5.4-nano` |",
    "| Judge / starker Reasoner | `judge_llm` | `o3` |",
    "| Worker / Synthese | `worker_llm` | `gpt-5.4-mini` |",
    "| Embeddings | `embed_model` | `text-embedding-3-small` |",
]
mprint("\n".join(zeilen))

# 3 | Docker-Container für Agenten
---

Docker kapselt den Agenten mit allen Abhängigkeiten in einen **isolierten Container**.
Damit läuft der Agent auf jedem System identisch — ob Laptop, Server oder Cloud.

**Drei Dateien für ein Production-Deployment:**

| Datei | Aufgaben |
|-------|---------|
| `Dockerfile` | Baut das Container-Image (Python, Pakete, Code) |
| `docker-compose.yml` | Orchestriert Container + Umgebungsvariablen |
| `.env` | API-Keys (nie ins Git-Repository!) |

```bash
**Typischer Workflow**
docker build -t mein-agent .          # Image bauen
docker-compose up -d                  # Container starten
curl http://localhost:8080/health      # Health-Check
docker-compose logs -f                # Logs verfolgen
docker-compose down                   # Container stoppen
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Docker Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    CLI(["🖥️ Client\ncurl / requests"])
    DC["🐳 Docker Container\nport 8080:8080"]

    subgraph container ["Container"]
        UV["uvicorn\nASGI Server"]
        FA["FastAPI App\nagent_server.py"]
        AG["LangChain Agent\ngenai_lib.briefing_rag"]
    end

    ENV[("🔑 .env\nOPENAI_API_KEY\nLANGSMITH_API_KEY")]
    LS["☁️ LangSmith\nTracing"]

    CLI -->|HTTP POST /invoke| DC
    DC --> UV --> FA --> AG
    ENV -.->|os.environ| DC
    AG -.->|Traces| LS

    style DC  fill:#1565C0,color:#fff
    style ENV fill:#4A148C,color:#fff
    style LS  fill:#2E7D32,color:#fff
    style CLI fill:#E65100,color:#fff
'''
mermaid(diagram, width=800)

<p><font color='darkblue' size="4">
🔗 <b>Interaktive Visualisierung</b>
</font></p>

[Docker](https://editor.p5js.org/ralf.bendig.rb/full/dbBJ9BhRP) visualisiert Containerisierung und Deployment-Workflow interaktiv — ergänzend zur Architektur-Grafik oben.

<p><font size="4" color='green'>  🐳 Dockerfile + docker-compose.yml generieren</font> </br></p>

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
httpx
git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul


In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Abhängigkeiten installieren
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Code kopieren
COPY agent_server.py ./

# Port freigeben
EXPOSE 8080

# Health-Check ohne zusätzliche curl-Abhängigkeit
HEALTHCHECK --interval=30s --timeout=5s \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health', timeout=3)" || exit 1

# Server starten
CMD ["uvicorn", "agent_server:app", "--host", "0.0.0.0", "--port", "8080"]

In [ ]:
%%writefile docker-compose.yml
version: '3.9'
services:
  agent:
    build: .
    ports:
      - "8080:8080"
    env_file:
      - .env
    volumes:
      - ./chroma_briefing:/data/chroma_briefing:ro
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8080/health', timeout=3)"]
      interval: 30s
      timeout: 5s
      retries: 3

In [ ]:
%%writefile .env.template
# API-Keys — NICHT ins Git-Repository!
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=prod-agent
LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com
# Muss zum gemounteten Volume in docker-compose.yml passen
BRIEFING_CHROMA_PATH=/data/chroma_briefing

In [ ]:
import os
zeilen = [
    "🐳 **Generierte Deployment-Dateien**", "",
    "| Datei | Größe | Zweck |",
    "|-------|-------|-------|",
]
for fname, desc in [
    ("Dockerfile",        "Container-Image Definition"),
    ("docker-compose.yml","Container-Orchestrierung"),
    (".env.template",     "API-Key Template (nicht committen!)"),
]:
    size = os.path.getsize(fname) if os.path.exists(fname) else 0
    zeilen.append(f"| `{fname}` | {size} Bytes | {desc} |")
mprint("\n".join(zeilen))

> Fortsetzung in **M35b — API, Monitoring & Kursrückblick**: FastAPI-Endpoints, Production Monitoring und der Abschluss-Überblick über die Agenten-Architektur.

Weiter in **M35b_API_Monitoring**: Dort wird das Deployment um FastAPI-Endpunkte, Monitoring und Betriebsbeobachtung erweitert.


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Aus Entwicklung ins Deployment](https://ralf-42.github.io/Agenten/08-deployment-betrieb/aus-entwicklung-ins-deployment.html)
- [Vom Modell zum Produkt](https://ralf-42.github.io/Agenten/08-deployment-betrieb/vom-modell-zum-produkt-langchain-oekosystem.html)
- [Migration-Analyse Provider](https://ralf-42.github.io/Agenten/08-deployment-betrieb/migration-openai-mistral.html)
- [Digitale Souveränität](https://ralf-42.github.io/Agenten/09-regulatorik-verantwortung/digitale-souveraenitaet.html)
